About Dataset
Context
Stable benchmark dataset. 20 million ratings and 465,000 tag applications applied to 27,000 movies by 138,000 users.

Content
this dataset has got three files named as ratings.csv, movies.csv and tags.csv

movies.csv
In the 3 columns stored are the values of movieId, title and genre. The title has got the release year of movie in parenthesis. The movie list range from Dickson Greeting (1891) to movies of 2015. With the total of 27278 movies.

ratings.csv
the movies have been rated by 138493 users on the scale of 1 to 5, this file contains the information divided in the column 'userId', 'movieId', 'rating' and 'timestamp'.

tags.csv
this file has the data divided under category 'userId','movieId' and 'tag'

Acknowledgements
I got this data from MovieLens, for a mini project.
This is the link to original data set

In [ ]:
#import the reqired libraries
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import numpy as np
import pandas as pd
import math
import json
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity # measure the similarity between two or more vectors.
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
import scipy.sparse
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
import warnings; warnings.simplefilter('ignore')
%matplotlib inline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
dataset1 = pd.read_csv('/content/drive/MyDrive/CapstoneProjectNetflix/movies.csv',header = 0, usecols = [0,1,2])
dataset1.head()
# Split the 'Genre' column into separate columns
df_genre = dataset1['genres'].str.get_dummies('|')

# Concatenate the new columns with the original DataFrame
df_movies = pd.concat([dataset1, df_genre], axis=1)

# Drop the original 'Genre' column
df_movies.drop(columns=['genres'], inplace=True)

# Display the modified Date
print(df_movies.head())
df_movies.shape

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


   movieId                               title  (no genres listed)  Action  \
0        1                    Toy Story (1995)                   0       0   
1        2                      Jumanji (1995)                   0       0   
2        3             Grumpier Old Men (1995)                   0       0   
3        4            Waiting to Exhale (1995)                   0       0   
4        5  Father of the Bride Part II (1995)                   0       0   

   Adventure  Animation  Children  Comedy  Crime  Documentary  ...  Film-Noir  \
0          1          1         1       1      0            0  ...          0   
1          1          0         1       0      0            0  ...          0   
2          0          0         0       1      0            0  ...          0   
3          0          0         0       1      0            0  ...          0   
4          0          0         0       1      0            0  ...          0   

   Horror  IMAX  Musical  Mystery  Romance  

(27278, 22)

In [ ]:
dataset2 = pd.read_csv('/content/drive/MyDrive/CapstoneProjectNetflix/ratings.csv',header = 0, usecols = [0,1,2,3])
dataset2.head()
dataset2.shape
dataset2 = dataset2.drop('timestamp', axis=1)
dataset2.head()

,userId,movieId,rating,timestamp
0,1,2,3.5,1112486027
1,1,29,3.5,1112484676
2,1,32,3.5,1112484819
3,1,47,3.5,1112484727
4,1,50,3.5,1112484580


(20000263, 4)

,userId,movieId,rating
0,1,2,3.5
1,1,29,3.5
2,1,32,3.5
3,1,47,3.5
4,1,50,3.5


In [ ]:
dataset3 = pd.read_csv('/content/drive/MyDrive/CapstoneProjectNetflix/tags.csv',header = 0, usecols = [0,1,2])
dataset3.head()
dataset3.shape
dataset3 = dataset3.drop('tag', axis=1)
dataset3.head()

,userId,movieId,tag
0,18,4141,Mark Waters
1,65,208,dark hero
2,65,353,dark hero
3,65,521,noir thriller
4,65,592,dark hero


(465564, 3)

,userId,movieId
0,18,4141
1,65,208
2,65,353
3,65,521
4,65,592


In [ ]:
merged_data = pd.merge(dataset2, dataset3, on=['movieId', 'userId'], how='inner')

# Display the merged DataFrame
print(merged_data.head())
merged_data.shape

   userId  movieId  rating
0      65    27866     4.0
1      65    48082     4.5
2      65    48082     4.5
3      65    58652     5.0
4      65    58652     5.0


(391445, 3)

In [ ]:
df = pd.merge(merged_data, df_movies, on='movieId', how='inner')
df.head()
df.shape

,userId,movieId,rating,title,(no genres listed),Action,Adventure,Animation,Children,Comedy,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,65,27866,4.0,In My Father's Den (2004),0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,20520,27866,4.0,In My Father's Den (2004),0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,20520,27866,4.0,In My Father's Den (2004),0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,53931,27866,3.5,In My Father's Den (2004),0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,71833,27866,3.5,In My Father's Den (2004),0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


(391445, 24)

In [ ]:
nan_values = df.isna().sum()
nan_values

userId                0
movieId               0
rating                0
title                 0
(no genres listed)    0
Action                0
Adventure             0
Animation             0
Children              0
Comedy                0
Crime                 0
Documentary           0
Drama                 0
Fantasy               0
Film-Noir             0
Horror                0
IMAX                  0
Musical               0
Mystery               0
Romance               0
Sci-Fi                0
Thriller              0
War                   0
Western               0
dtype: int64

In [ ]:
df1 = df[df['(no genres listed)'] != 1]
df2 = df1.drop('(no genres listed)', axis=1)
df2.head()
df2.shape

,userId,movieId,rating,title,Action,Adventure,Animation,Children,Comedy,Crime,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,65,27866,4.0,In My Father's Den (2004),0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,20520,27866,4.0,In My Father's Den (2004),0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,20520,27866,4.0,In My Father's Den (2004),0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,53931,27866,3.5,In My Father's Den (2004),0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,71833,27866,3.5,In My Father's Den (2004),0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


(391399, 23)

In [ ]:
df1['rating'].min()

0.5

In [ ]:
df1['rating'].max()

5.0

In [ ]:
df2.describe()

,userId,movieId,rating,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
count,391399.000000,391399.000000,391399.000000,391399.000000,391399.000000,391399.000000,391399.000000,391399.000000,391399.000000,391399.000000,...,391399.000000,391399.000000,391399.000000,391399.000000,391399.000000,391399.000000,391399.000000,391399.000000,391399.000000,391399.000000
mean,67292.121362,32755.404866,3.780701,0.274209,0.197239,0.060409,0.052509,0.303813,0.174487,0.023076,...,0.015043,0.092639,0.063843,0.033462,0.108570,0.174599,0.200243,0.293976,0.050036,0.014326
std,42228.094963,35985.878041,1.024126,0.446115,0.397915,0.238243,0.223052,0.459903,0.379528,0.150146,...,0.121726,0.289927,0.244473,0.179840,0.311099,0.379625,0.400183,0.455582,0.218019,0.118829
min,65.000000,1.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,27898.000000,2502.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,66635.000000,7361.000000,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,106755.000000,63072.000000,4.500000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
max,138472.000000,131258.000000,5.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [ ]:
print(df2.columns)

Index(['userId', 'movieId', 'rating', 'title', 'Action', 'Adventure',
       'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama',
       'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery',
       'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'],
      dtype='object')


1. Find out the list of most popular and liked genre?

In [ ]:
import pandas as pd

# Find the 75th percentile values
percentile_75 = df.describe().loc['75%']

# Extract column names where the value is 1.0 in the 75th percentile row
columns_with_1 = percentile_75[percentile_75 == 1.0].index

# Display the column names
print(columns_with_1)

Index(['Action', 'Comedy', 'Drama', 'Thriller'], dtype='object')


In [ ]:
import pandas as pd

# Most rated genre -popular
genre_columns = df2.columns[4:]
# Count the number of ratings for each genre
mostrated = df2[genre_columns].sum()
# Display the genres with the highest number of ratings
mostrated = mostrated.sort_values(ascending=False)

df3=df2.copy()
# most liked genre -highrated 4,5
genre_columns = df3.columns[4:]
df3['rating'] = df3['rating'].round().astype(int)
df_rating_5  = df3[df3['rating'].isin([4, 5])]
highrated = df_rating_5[genre_columns].sum()
highrated = highrated.sort_values(ascending=False)

# Find the common genres between most liked and most popular
common_genres = mostrated.index.intersection(highrated.index).intersection(columns_with_1)

print("\nmost liked and most popular genre:")
print(common_genres)


most liked and most popular genre:
Index(['Drama', 'Comedy', 'Thriller', 'Action'], dtype='object')


2.Create Model that finds the best suited Movie for one
user in every genre?

In [ ]:
df3.head()

,userId,movieId,rating,title,Action,Adventure,Animation,Children,Comedy,Crime,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,65,27866,4,In My Father's Den (2004),0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,20520,27866,4,In My Father's Den (2004),0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,20520,27866,4,In My Father's Den (2004),0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,53931,27866,4,In My Father's Den (2004),0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,71833,27866,4,In My Father's Den (2004),0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

# Assume your dataset is named 'ratings_data'
# Columns: userId, movieId, rating, title, Action, Adventure, Animation, ..., Western
ratings_data = df3.copy()
# Split the data into training and testing sets
train_data, test_data = train_test_split(ratings_data, test_size=0.2, random_state=42)

# Create a user-movie matrix
user_movie_matrix = train_data.pivot_table(index='userId', columns='movieId', values='rating')

# Fill missing values with 0 (assuming no rating means a rating of 0)
user_movie_matrix = user_movie_matrix.fillna(0)

# Calculate cosine similarity between users
user_similarity = cosine_similarity(user_movie_matrix)

# Function to get movie recommendations for a user in every genre
def get_movie_recommendations_for_all_genres(user_id):
    all_genres = list(ratings_data.columns[5:])  # Assuming genres start from the 6th column

    all_recommendations = pd.DataFrame(columns=['genre', 'movieId', 'score', 'rating', 'title'])

    for genre in all_genres:
        genre_data = ratings_data[ratings_data[genre] == 1]

        # Get movies the user has not watched in this genre
        user_watched_movies = genre_data[genre_data['userId'] == user_id]['movieId']
        all_movies = set(genre_data['movieId'])
        movies_to_recommend = list(all_movies - set(user_watched_movies))

        # Calculate weighted average rating based on user similarity
        user_similarities = user_similarity[user_id - 1]
        weighted_ratings = user_movie_matrix.values * user_similarities.reshape(-1, 1)

        # Sum up the weighted ratings and normalize
        movie_scores = weighted_ratings.sum(axis=0) / (user_similarities.sum() + 1e-10)

        # Create a dataframe with movieId and scores
        recommendations_df = pd.DataFrame({'movieId': user_movie_matrix.columns, 'score': movie_scores})

        # Filter recommendations by genre
        genre_recommendations = recommendations_df[recommendations_df['movieId'].isin(movies_to_recommend)]

        # Sort and get the top recommendation for this genre
        top_recommendation = genre_recommendations.groupby('movieId').max().sort_values(by='score', ascending=False).head(1)

        # Merge with movie data to get additional information
        final_recommendation = pd.merge(top_recommendation, genre_data.drop_duplicates('movieId'), on='movieId')
        final_recommendation['genre'] = genre

        # Append to the overall recommendations dataframe
        all_recommendations = pd.concat([all_recommendations, final_recommendation], ignore_index=True)

    return all_recommendations[['genre', 'movieId', 'score', 'rating', 'title']]

# Example usage
user_id_to_recommend = 60

recommendations_for_all_genres = get_movie_recommendations_for_all_genres(user_id_to_recommend)
print(recommendations_for_all_genres)


          genre movieId     score rating  \
0     Adventure   72998  0.381019      4   
1     Animation    5618  0.371992      5   
2      Children   60069  0.370968      4   
3        Comedy   79702  1.184324      5   
4         Crime    2959  1.135550      5   
5   Documentary   44788  0.112688      4   
6         Drama    4878  3.327435      5   
7       Fantasy   79702  1.184324      5   
8     Film-Noir   32587  0.510411      4   
9        Horror     593  0.482776      5   
10         IMAX   79132  0.787094      4   
11      Musical   79702  1.184324      5   
12      Mystery    4878  3.327435      5   
13      Romance   79702  1.184324      5   
14       Sci-Fi    4878  3.327435      5   
15     Thriller    4878  3.327435      5   
16          War     356  0.426327      4   
17      Western   99114  0.183941      4   

                                                title  
0                                       Avatar (2009)  
1   Spirited Away (Sen to Chihiro no kamikakushi) .

3.Find what Genre Movies have received the best and
worst ratings based on User Rating?

In [ ]:
import pandas as pd

# Group by title and rating, then unstack to get count for each rating
rating_counts_by_movie = df3.groupby(['title', 'rating']).size().unstack(fill_value=0).reset_index()
rating_counts_by_movie.columns = ['title', 'count_rating_0', 'count_rating_1', 'count_rating_2', 'count_rating_3', 'count_rating_4', 'count_rating_5']

# Find movies with the maximum count for each rating
max_counts_by_rating = rating_counts_by_movie.loc[rating_counts_by_movie[['count_rating_0', 'count_rating_1', 'count_rating_2', 'count_rating_3', 'count_rating_4', 'count_rating_5']].idxmax()]

# Merge with original data to get additional information
combined_data = pd.merge(max_counts_by_rating, dataset1[['title', 'movieId', 'genres']], on='title', how='inner')

# Find the movie with the maximum count for rating 5
max_count_rating_5 = combined_data['count_rating_5'].max()
max_count_rating_5_movie = combined_data[combined_data['count_rating_5'] == max_count_rating_5][['title', 'genres']]
print("Movie with the maximum count_rating_5:")
print(max_count_rating_5_movie)

# Find the movie with the maximum count for rating 0
max_count_rating_0 = combined_data['count_rating_0'].max()
max_count_rating_0_movie = combined_data[combined_data['count_rating_0'] == max_count_rating_0][['title', 'genres']]
print("\nMovie with the maximum count_rating_0:")
print(max_count_rating_0_movie)


Movie with the maximum count_rating_5:
               title                       genres
5  Fight Club (1999)  Action|Crime|Drama|Thriller

Movie with the maximum count_rating_0:
                                      title       genres
0  Expelled: No Intelligence Allowed (2008)  Documentary
